# MambaIRv2 TPAF Denoising — Training Notebook (Phases 1–4)

This notebook trains **MambaIRv2** (CVPR 2025) as a denoising model for two-photon autofluorescence (TPAF) microscopy images.

**Workflow:**
1. Mount Google Drive
2. Clone the MambaIR repository & install all dependencies
3. Copy training data from Drive to local Colab storage (faster I/O)
4. Auto-detect `in_chans` and `model_type` from the actual data & repo
5. Programmatically generate the training YAML config
6. Symlink the `experiments/` directory to Drive (checkpoint persistence)
7. Start training

A **separate resume section** at the bottom lets you restart after a Colab disconnect.

## 0  User Configuration
Edit **only this cell** before running the notebook.

In [ ]:
# ============================================================
# USER CONFIGURATION — edit these values before running
# ============================================================

# --- Google Drive paths ---
DRIVE_DATA_ROOT  = "/content/drive/MyDrive/TPAF_data"
# ^ Must contain:  train/clean/  train/noisy/  val/clean/  val/noisy/

DRIVE_EXPR_ROOT  = "/content/drive/MyDrive/MambaIR_experiments"
# ^ Checkpoints are symlinked here so they survive Colab disconnects.

# --- Local (Colab) paths ---
LOCAL_DATA_ROOT  = "/content/data"    # fast local SSD for data
MAMBAIR_DIR      = "/content/MambaIR" # where the repo is cloned

# --- Training hyper-parameters ---
BATCH_SIZE  = 4        # per GPU; reduce to 2 if OOM
GT_SIZE     = 128      # training crop size (img_size in config must match)
TOTAL_ITER  = 400000   # total training iterations
NUM_GPU     = 1        # number of GPUs
MANUAL_SEED = 42

LR          = "2e-4"   # learning rate (kept as string for YAML !!float tag)

# --- Model architecture (leave as-is for MambaIRv2-Base) ---
EMBED_DIM = 180
DEPTHS    = [6, 6, 6, 6, 6, 6]
NUM_HEADS = [6, 6, 6, 6, 6, 6]

# ============================================================
print("Configuration loaded.")
print(f"  Drive data root : {DRIVE_DATA_ROOT}")
print(f"  Drive expr root : {DRIVE_EXPR_ROOT}")
print(f"  Local data root : {LOCAL_DATA_ROOT}")
print(f"  MambaIR dir     : {MAMBAIR_DIR}")
print(f"  Batch size      : {BATCH_SIZE}  |  GT crop : {GT_SIZE}  |  Total iter : {TOTAL_ITER}")

## 1  Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## 2  Clone the MambaIR Repository

The **main branch** of [csguoh/MambaIR](https://github.com/csguoh/MambaIR) contains MambaIRv2.

In [ ]:
import os, subprocess

if not os.path.exists(MAMBAIR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/csguoh/MambaIR.git", MAMBAIR_DIR],
        check=True,
    )
    print(f"Cloned MambaIR to {MAMBAIR_DIR}")
else:
    print(f"Repo already exists at {MAMBAIR_DIR} — skipping clone.")

os.chdir(MAMBAIR_DIR)
print(f"Working directory: {os.getcwd()}")

## 3  Install Dependencies (pip only)

> **Note:** `mamba_ssm` compilation takes ~3–5 minutes on Colab.
> If `mamba_ssm==1.1.1` fails to build, the cell tries the latest version from source.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# 1) PyTorch with CUDA 11.8 wheels (compatible with Colab's CUDA runtime)
pip("torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2",
    "--index-url", "https://download.pytorch.org/whl/cu118")

# 2) Mamba SSM dependencies
try:
    pip("causal_conv1d==1.1.1")
    pip("mamba_ssm==1.1.1")
    print("mamba_ssm 1.1.1 installed successfully.")
except Exception:
    print("Fixed-version install failed — trying latest from source...")
    pip("causal-conv1d>=1.1.0")
    pip("mamba-ssm", "--no-build-isolation")

# 3) BasicSR + extras
pip("basicsr==1.3.5", "einops", "timm")

# 4) Install the MambaIR repo itself (editable)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

print("\nAll dependencies installed.")

## 4  Copy Data from Google Drive to Local Storage

Copying to `/content/` gives ~10× faster I/O compared with reading directly from Drive.

In [ ]:
import shutil, os

LOCAL_TRAIN_CLEAN = f"{LOCAL_DATA_ROOT}/train/clean"
LOCAL_TRAIN_NOISY = f"{LOCAL_DATA_ROOT}/train/noisy"
LOCAL_VAL_CLEAN   = f"{LOCAL_DATA_ROOT}/val/clean"
LOCAL_VAL_NOISY   = f"{LOCAL_DATA_ROOT}/val/noisy"

pairs = [
    (f"{DRIVE_DATA_ROOT}/train/clean", LOCAL_TRAIN_CLEAN),
    (f"{DRIVE_DATA_ROOT}/train/noisy", LOCAL_TRAIN_NOISY),
    (f"{DRIVE_DATA_ROOT}/val/clean",   LOCAL_VAL_CLEAN),
    (f"{DRIVE_DATA_ROOT}/val/noisy",   LOCAL_VAL_NOISY),
]

for src, dst in pairs:
    if os.path.exists(dst):
        print(f"Already exists — skipping: {dst}")
    else:
        print(f"Copying {src}  →  {dst} ...")
        shutil.copytree(src, dst)
        print(f"  Done.")

print("\nData copy complete.")

## 5  Verify Data Alignment

Clean and noisy filenames **must be identical and in the same order**.

In [ ]:
train_clean = sorted(f for f in os.listdir(LOCAL_TRAIN_CLEAN) if not f.startswith('.'))
train_noisy = sorted(f for f in os.listdir(LOCAL_TRAIN_NOISY) if not f.startswith('.'))
val_clean   = sorted(f for f in os.listdir(LOCAL_VAL_CLEAN)   if not f.startswith('.'))
val_noisy   = sorted(f for f in os.listdir(LOCAL_VAL_NOISY)   if not f.startswith('.'))

assert len(train_clean) == len(train_noisy), (
    f"Train count mismatch: {len(train_clean)} clean vs {len(train_noisy)} noisy")
assert train_clean == train_noisy, (
    f"Train filename mismatch!\n  First diff: {next((a,b) for a,b in zip(train_clean,train_noisy) if a!=b)}")
assert len(val_clean) == len(val_noisy), (
    f"Val count mismatch: {len(val_clean)} clean vs {len(val_noisy)} noisy")
assert val_clean == val_noisy, "Val filename mismatch!"

print(f"Train pairs : {len(train_clean)}")
print(f"Val pairs   : {len(val_clean)}")
print("Data alignment check PASSED.")

## 6  Auto-detect `in_chans`

BasicSR's `imread` may silently convert grayscale images to 3-channel RGB.
We load one sample through the actual data pipeline to discover the real channel count.

In [ ]:
import sys
sys.path.insert(0, MAMBAIR_DIR)
from basicsr.data.paired_image_dataset import PairedImageDataset

_opt = {
    "dataroot_gt"   : LOCAL_TRAIN_CLEAN,
    "dataroot_lq"   : LOCAL_TRAIN_NOISY,
    "filename_tmpl" : "{}",
    "io_backend"    : {"type": "disk"},
    "gt_size"       : 128,
    "use_hflip"     : False,
    "use_rot"       : False,
    "phase"         : "train",
    "scale"         : 1,
}
_ds     = PairedImageDataset(_opt)
_sample = _ds[0]
IN_CHANS = int(_sample["gt"].shape[0])   # 1 = grayscale kept as-is, 3 = auto-converted to RGB
del _ds, _sample, _opt

print(f"Detected in_chans = {IN_CHANS}")
if IN_CHANS == 1:
    print("  → Grayscale images loaded as single-channel (ideal).")
else:
    print("  → BasicSR converted grayscale to 3-channel RGB.")
    print("    The config will use in_chans=3 accordingly.")

## 7  Auto-detect `model_type`

The correct `model_type` string is read from the existing MambaIR denoising configs so our YAML is always consistent with the installed codebase.

In [ ]:
import glob, re

MODEL_TYPE = None
search_patterns = [
    f"{MAMBAIR_DIR}/options/train/mambairv2/*DN*.yml",
    f"{MAMBAIR_DIR}/options/train/mambairv2/*denois*.yml",
    f"{MAMBAIR_DIR}/options/train/mambairv2/*.yml",
]

for pattern in search_patterns:
    for yml_path in sorted(glob.glob(pattern)):
        with open(yml_path) as fh:
            text = fh.read()
        m = re.search(r'^model_type:\s*(\S+)', text, re.MULTILINE)
        if m:
            MODEL_TYPE = m.group(1)
            print(f"Found model_type '{MODEL_TYPE}'")
            print(f"  Source: {yml_path}")
            break
    if MODEL_TYPE:
        break

if MODEL_TYPE is None:
    MODEL_TYPE = "ImageCleanModel"
    print(f"model_type not found in existing configs — defaulting to '{MODEL_TYPE}'")

print(f"\nUsing model_type = {MODEL_TYPE}")

## 8  Generate Training YAML Config

The config is written programmatically using the auto-detected `IN_CHANS` and `MODEL_TYPE` values.

In [ ]:
import os

yml_dir        = f"{MAMBAIR_DIR}/options/train/mambairv2"
TRAIN_YML_PATH = f"{yml_dir}/train_MambaIRv2_TPAF_denoise.yml"
os.makedirs(yml_dir, exist_ok=True)

TRAIN_YML = f"""# -----------------------------------------------------------------------
# MambaIRv2 Denoising — TPAF Microscopy (auto-generated by train.ipynb)
# -----------------------------------------------------------------------
name: MambaIRv2_TPAF_Denoise
model_type: {MODEL_TYPE}
scale: 1
num_gpu: {NUM_GPU}
manual_seed: {MANUAL_SEED}

# ---------- Datasets ----------
datasets:
  train:
    task: DN
    name: TPAF_train
    type: PairedImageDataset

    dataroot_gt: {LOCAL_TRAIN_CLEAN}
    dataroot_lq: {LOCAL_TRAIN_NOISY}
    filename_tmpl: '{{}}'

    io_backend:
      type: disk

    gt_size: {GT_SIZE}
    use_hflip: true
    use_rot: true
    use_shuffle: true
    num_worker_per_gpu: 4
    batch_size_per_gpu: {BATCH_SIZE}
    dataset_enlarge_ratio: 1
    prefetch_mode: ~

  val:
    task: DN
    name: TPAF_val
    type: PairedImageDataset

    dataroot_gt: {LOCAL_VAL_CLEAN}
    dataroot_lq: {LOCAL_VAL_NOISY}
    filename_tmpl: '{{}}'

    io_backend:
      type: disk

# ---------- Network ----------
network_g:
  type: MambaIRv2
  upscale: 1
  in_chans: {IN_CHANS}
  img_size: {GT_SIZE}
  img_range: 1.0
  embed_dim: {EMBED_DIM}
  d_state: 8
  depths: {DEPTHS}
  num_heads: {NUM_HEADS}
  window_size: 16
  inner_rank: 32
  num_tokens: 64
  convffn_kernel_size: 5
  mlp_ratio: 2
  upsampler: ''
  resi_connection: '1conv'

# ---------- Paths ----------
path:
  pretrain_network_g: ~
  strict_load_g: true
  resume_state: ~

# ---------- Training ----------
train:
  ema_decay: 0.999

  optim_g:
    type: Adam
    lr: !!float {LR}
    weight_decay: 0
    betas: [0.9, 0.999]

  scheduler:
    type: MultiStepLR
    milestones: [100000, 200000, 300000, 350000]
    gamma: 0.5

  total_iter: {TOTAL_ITER}
  warmup_iter: -1

  pixel_opt:
    type: CharbonnierLoss
    loss_weight: 1.0
    reduction: mean

# ---------- Validation ----------
val:
  val_freq: !!float 5e3
  save_img: false

  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 0
      test_y_channel: false
    ssim:
      type: calculate_ssim
      crop_border: 0
      test_y_channel: false

# ---------- Logging ----------
logger:
  print_freq: 200
  save_checkpoint_freq: !!float 5e3
  use_tb_logger: true
  wandb:
    project: ~
    resume_id: ~

# ---------- Distributed ----------
dist_params:
  backend: nccl
"""

with open(TRAIN_YML_PATH, "w") as fh:
    fh.write(TRAIN_YML)

print(f"Training config written to:")
print(f"  {TRAIN_YML_PATH}")
print(f"\nKey settings:")
print(f"  model_type = {MODEL_TYPE}")
print(f"  in_chans   = {IN_CHANS}")
print(f"  gt_size    = {GT_SIZE}  (img_size matches)")
print(f"  batch_size = {BATCH_SIZE}")
print(f"  total_iter = {TOTAL_ITER}")


## 9  Symlink `experiments/` to Google Drive

Colab VMs reset on disconnect. By symlinking the `experiments/` directory to your Drive, **checkpoints are written directly to Drive** and survive disconnects.

In [ ]:
import os, shutil

LOCAL_EXPR_PATH = f"{MAMBAIR_DIR}/experiments"

# Ensure the Drive target directory exists
os.makedirs(DRIVE_EXPR_ROOT, exist_ok=True)

if os.path.islink(LOCAL_EXPR_PATH):
    current_target = os.readlink(LOCAL_EXPR_PATH)
    print(f"Symlink already exists: {LOCAL_EXPR_PATH} → {current_target}")

elif os.path.isdir(LOCAL_EXPR_PATH):
    print(f"{LOCAL_EXPR_PATH} exists as a real directory — migrating to Drive ...")
    shutil.copytree(LOCAL_EXPR_PATH, DRIVE_EXPR_ROOT, dirs_exist_ok=True)
    shutil.rmtree(LOCAL_EXPR_PATH)
    os.symlink(DRIVE_EXPR_ROOT, LOCAL_EXPR_PATH)
    print(f"Migrated and symlinked: {LOCAL_EXPR_PATH} → {DRIVE_EXPR_ROOT}")

else:
    os.symlink(DRIVE_EXPR_ROOT, LOCAL_EXPR_PATH)
    print(f"Symlinked: {LOCAL_EXPR_PATH} → {DRIVE_EXPR_ROOT}")

print("\nCheckpoints will be saved to Drive automatically.")

## 10  TensorBoard Monitoring (optional)

Run this cell **before or after** starting training to view live loss and PSNR curves.
The TensorBoard widget updates automatically while training runs in the cell below.

In [ ]:
import os
TB_LOG_DIR = f"{MAMBAIR_DIR}/experiments/MambaIRv2_TPAF_Denoise/tb_logger"
os.makedirs(TB_LOG_DIR, exist_ok=True)

%load_ext tensorboard
%tensorboard --logdir {TB_LOG_DIR}

## 11  Start Training

This cell runs training from scratch.
**If you need to resume after a disconnect, skip to the [Resume Section](#resume) below.**

> Expected behaviour:
> - `l_pix` loss should decrease from ~0.05 toward ~0.005–0.01
> - Validation PSNR should increase; good denoising results are ≥30 dB
> - Checkpoints saved every 5 000 iterations to Drive

In [ ]:
import os
os.chdir(MAMBAIR_DIR)

# Sanity-check that the config exists
assert os.path.exists(TRAIN_YML_PATH), f"Config not found: {TRAIN_YML_PATH}"
print(f"Starting training with config: {TRAIN_YML_PATH}\n")

!python basicsr/train.py -opt {TRAIN_YML_PATH}

---
<a id="resume"></a>
## 12  Resume Training After a Colab Disconnect

After a Colab session resets:

1. **Re-run cells 0–9** (Configuration → Symlink `experiments/`) to restore the environment.
   Data copy (cell 4) will be skipped automatically if the local copies already exist.
2. **Run the cell below** — `--auto_resume` picks up the latest checkpoint from the
   `experiments/MambaIRv2_TPAF_Denoise/training_states/` directory automatically.

> **Tip:** You can also set `resume_state` manually in the generated YAML to a specific
> `.state` file if you want to resume from a particular checkpoint.

In [ ]:
import os
os.chdir(MAMBAIR_DIR)

# Verify checkpoint directory on Drive
import glob
state_files = sorted(glob.glob(
    f"{DRIVE_EXPR_ROOT}/MambaIRv2_TPAF_Denoise/training_states/*.state"
))
if state_files:
    print(f"Found {len(state_files)} training state(s). Latest:")
    print(f"  {state_files[-1]}")
else:
    print("No training states found — will start from scratch if --auto_resume finds nothing.")

print(f"\nResuming training ...\n")
!python basicsr/train.py -opt {TRAIN_YML_PATH} --auto_resume